# K03_05 – Grid Search und Hyperparameter – Dozentenversion

Letzter Update am 23. Mai 2026

Diese Fassung enthält **Musterlösungen und kurze didaktische Hinweise**.

Dieses Notebook ist die **Vertiefung** zu Kapitel 3.  
Wir suchen systematisch gute Hyperparameter mit `GridSearchCV`.

## Lernziele
Nach diesem Notebook können Sie:
- den Begriff **Hyperparameter** erklären
- ein kleines Parametersuchgitter definieren
- `GridSearchCV` anwenden
- die Ergebnisse der Suche interpretieren

## 1. Problem und Datensatz

Wir bleiben beim Iris-Datensatz, damit der Fokus auf der Methode liegt.


In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

## 2. Basismodell


In [4]:
baseline = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier()
)

baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
acc_base = accuracy_score(y_test, y_pred_base)

print(f"Baseline-Accuracy: {acc_base:.3f}")

Baseline-Accuracy: 0.921


## 3. Was ist ein Hyperparameter?

In scikit-learn gibt es zwei Arten von Werten:

| | **Parameter** | **Hyperparameter** |
|---|---|---|
| Wer legt ihn fest? | Das Modell (beim Training) | Wir – vor dem Training |
| Wie? | Durch Optimierung auf Daten | Durch Suche / Ausprobieren |
| Beispiel k-NN | (k-NN speichert Trainingsdaten) | `n_neighbors`, `weights` |
| Beispiel LinReg | `coef_`, `intercept_` | (wenige) |

Bei k-NN ist `n_neighbors` ein typischer Hyperparameter:  
Er wird **nicht aus den Daten gelernt**, sondern von uns festgelegt.

> **Merksatz:** Parameter lernt das Modell. Hyperparameter legen wir fest.
> `GridSearchCV` hilft uns, gute Hyperparameter **systematisch** zu finden.


## 4. Grid Search vorbereiten

Wir definieren ein **Parametergitter**: alle Werte, die GridSearchCV ausprobieren soll.

### Die `__`-Syntax im param_grid

Da wir eine Pipeline verwenden, geben wir an, **welcher Schritt** den Parameter hat:
```
"kneighborsclassifier__n_neighbors"
  ^-- Name des Pipeline-Schritts   ^-- Parametername
  (automatisch von make_pipeline)   (doppelter Unterstrich als Trennzeichen)
```

### Wie viele Kombinationen werden getestet?

```
n_neighbors: 6 Werte  x  weights: 2 Werte  =  12 Kombinationen
12 Kombinationen  x  cv=5 Folds            =  60 Trainingslaeufe
```

> GridSearchCV ist intern eine Schleife über alle Kombinationen
> mit Cross-Validation als Bewertungsmassstab.


In [7]:
pipeline = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier()
)

param_grid = {
    # Syntax: 'pipeline_schritt__parametername'
    # 'kneighborsclassifier' wird von make_pipeline automatisch vergeben
    "kneighborsclassifier__n_neighbors": [1, 3, 5, 7, 9, 11],  # 6 Werte
    "kneighborsclassifier__weights":     ["uniform", "distance"]  # 2 Werte
}  # -> 6 x 2 = 12 Kombinationen, jede mit cv=5 -> 60 Trainingslaeufe

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,                   # 5-fache Kreuzvalidierung pro Kombination
    scoring="accuracy",     # Bewertungsmassstab
    return_train_score=True # Trainingsscore zum Vergleich
)

n_k = len(param_grid["kneighborsclassifier__n_neighbors"])
n_w = len(param_grid["kneighborsclassifier__weights"])
print(f"Gittergroesse: {n_k} x {n_w} = {n_k * n_w} Kombinationen")
print(f"Trainingslaeufe gesamt: {n_k * n_w * 5}")


Gittergroesse: 6 x 2 = 12 Kombinationen
Trainingslaeufe gesamt: 60


## Mini-Übung 1
Bevor Sie die Suche starten:  
Welche Kombination vermuten Sie als sinnvoll – wenige oder viele Nachbarn? `uniform` oder `distance`?


### Musterlösung / Dozentenhinweis
Eine plausible Vermutung wäre:
- eher **mittlere** Zahlen von Nachbarn statt extrem kleiner oder extrem großer Werte
- `uniform` und `distance` sind beide sinnvoll testbar; `distance` kann hilfreich sein, wenn nahe Nachbarn stärker gewichtet werden sollen

Wichtig ist hier nicht die Vorhersage 'richtig' zu treffen, sondern die Idee: **Hyperparameter werden nicht geraten, sondern systematisch überprüft**.

## 5. Grid Search durchfuehren

### GridSearchCV und Cross-Validation -- die Verbindung

GridSearchCV verwendet intern **exakt dasselbe Prinzip** wie `cross_val_score`
 -- nur in einer Schleife über alle Parameterkombinationen:

```
Fuer jede der 12 Kombinationen:
    Fuer jeden der 5 Folds:
        1. Pipeline auf 4 Folds trainieren
        2. Pipeline auf dem 5. Fold bewerten
        3. Zwischenmodell verwerfen
    -> mittlerer CV-Score fuer diese Kombination

-> Beste Kombination = hoechster mittlerer CV-Score
-> GridSearchCV trainiert das beste Modell automatisch auf allen X_train
   und speichert es in best_estimator_
```

> **Merksatz:** GridSearchCV ist Cross-Validation in einer Schleife --
> mit automatischer Auswahl und Speicherung des besten Modells.


In [8]:
grid.fit(X_train, y_train)   # 60 Trainingslaeufe intern

print("Beste Parameter:")
print(grid.best_params_)
print()
print(f"Bester mittlerer CV-Score: {grid.best_score_:.3f}")
print()
print("Hinweis: Dieser CV-Score ist leicht optimistisch --")
print("er wurde genutzt, um die Parameter zu WAEHLEN (Selection Bias).")
print("Der ehrliche Score folgt in Schritt 7 auf den Testdaten.")


Beste Parameter:
{'kneighborsclassifier__n_neighbors': 7, 'kneighborsclassifier__weights': 'uniform'}

Bester mittlerer CV-Score: 0.955

Hinweis: Dieser CV-Score ist leicht optimistisch --
er wurde genutzt, um die Parameter zu WAEHLEN (Selection Bias).
Der ehrliche Score folgt in Schritt 7 auf den Testdaten.


## 6. Ergebnisse als Tabelle

Die Tabelle zeigt alle getesteten Kombinationen sortiert nach Rang:

| Spalte | Bedeutung |
|---|---|
| `mean_test_score` | Mittlerer CV-Score ueber alle 5 Folds |
| `std_test_score` | Streuung der Fold-Scores (Stabilitaet) |
| `rank_test_score` | Rang 1 = beste Kombination |

> **Hinweis:** Nicht nur `mean_test_score` zaehlt -- auch `std_test_score`.
> Ein Modell mit hohem Mittelwert aber hoher Streuung ist weniger zuverlaessig
> als eines mit etwas niedrigerem Mittelwert aber stabilen Fold-Scores.


In [9]:
results = pd.DataFrame(grid.cv_results_)
cols = [
    "param_kneighborsclassifier__n_neighbors",
    "param_kneighborsclassifier__weights",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]
results[cols].sort_values("rank_test_score").head(10)


,param_kneighborsclassifier__n_neighbors,param_kneighborsclassifier__weights,mean_test_score,std_test_score,rank_test_score
6,7,uniform,0.954941,0.040663,1
7,7,distance,0.954941,0.040663,1
9,9,distance,0.954941,0.040663,1
8,9,uniform,0.954941,0.040663,1
11,11,distance,0.954941,0.040663,1
10,11,uniform,0.954941,0.040663,1
0,1,uniform,0.946245,0.018598,7
1,1,distance,0.946245,0.018598,7
4,5,uniform,0.946245,0.034239,7
5,5,distance,0.946245,0.034239,7


## 7. Bestes Modell auf den Testdaten bewerten

### Was ist `best_estimator_`?

`grid.best_estimator_` ist das **fertig trainierte finale Modell** --
GridSearchCV hat automatisch:

1. die beste Parameterkombination identifiziert (via CV-Scores)
2. das Modell mit diesen Parametern auf **allen Trainingsdaten** neu trainiert
3. in `best_estimator_` gespeichert -- direkt einsatzbereit

Das entspricht dem Workflow aus Kapitel 3, den wir manuell durchgefuehrt haben:
```
cross_val_score -> besten k waehlen -> final_model.fit(X_train, y_train)
```
GridSearchCV erledigt das vollstaendig automatisch.

### Selection Bias -- warum der Test-Score jetzt wichtig ist

Der CV-Score aus `grid.best_score_` ist **leicht optimistisch**:
Wir haben ihn genutzt, um den besten Parameter zu *waehlen* --
das ist ein indirektes Hinschauen auf die Validierungsdaten.

Der **Test-Score** auf `X_test` ist die ehrliche, unabhaengige Schaetzung.
`X_test` wurde in keinem einzigen der 60 Trainingslaeufe verwendet.


In [10]:
best_model = grid.best_estimator_   # fertig trainiertes Modell auf allen X_train
y_pred_best = best_model.predict(X_test)
acc_best = accuracy_score(y_test, y_pred_best)

print(f"Bester CV-Score (leicht optimistisch): {grid.best_score_:.3f}")
print(f"Test-Accuracy bestes Modell (ehrlich): {acc_best:.3f}")
print(f"Test-Accuracy Baseline:                {acc_base:.3f}")
print()
if acc_best > acc_base:
    print(f"-> GridSearch verbessert die Baseline um {acc_best - acc_base:.3f}")
elif acc_best == acc_base:
    print("-> GridSearch liefert dasselbe Ergebnis wie die Baseline")
    print("   (auf kleinen Datensaetzen wie Iris ist das normal)")
else:
    print("-> Baseline ist besser -- kann bei kleinen Datensaetzen vorkommen")


Bester CV-Score (leicht optimistisch): 0.955
Test-Accuracy bestes Modell (ehrlich): 0.947
Test-Accuracy Baseline:                0.921

-> GridSearch verbessert die Baseline um 0.026


## Mini-Übung 2
1. Ist das beste Modell auf den Testdaten besser als die Baseline?
2. Warum ist es wichtig, die Hyperparametersuche **nicht** direkt auf den Testdaten zu machen?


### Musterloesung / Dozentenhinweis

**Frage 1:** Ob das beste Modell besser ist als die Baseline, zeigt der Vergleich
der Test-Scores. Auf kleinen Datensaetzen wie Iris sind die Unterschiede oft minimal.

**Frage 2:** Die Hyperparametersuche darf nicht auf den Testdaten stattfinden,
weil der Test-Set nur fuer die **einmalige, unabhängige Schlussbewertung** gedacht ist.
Sonst entsteht **Selection Bias**: die Parameter werden so gewählt,
dass sie auf diesen spezifischen Testdaten gut funktionieren --
das ist keine faire Schaetzung fuer neue, unbekannte Daten.

**Zusammenhang CV-Score und Test-Score:**
- `grid.best_score_` = mittlerer CV-Score der besten Kombination
  (leicht optimistisch -- wurde zur Parameterwahl genutzt)
- `accuracy_score(y_test, y_pred_best)` = Test-Score
  (ehrliche, unabhaengige Schaetzung)

Merksatz: *CV-Score waehlt das Modell. Test-Score bewertet es.*


## 8. Zusammenfassung

### Der vollstaendige GridSearchCV-Workflow

```
X_train, X_test = train_test_split(X, y)
         |
         +-- GridSearchCV(cv=5)
               |
               +-- Kombination 1: cross_val_score -> mittlerer Score
               +-- Kombination 2: cross_val_score -> mittlerer Score
               |   ...
               +-- Beste Kombination -> fit auf allen X_train
                                     -> best_estimator_ (einsatzbereit)
         |
         +-- best_estimator_.predict(X_test) -> ehrlicher Test-Score
```

### Merksaetze
- **Hyperparameter** legen wir fest -- **Parameter** lernt das Modell
- `GridSearchCV` ist Cross-Validation in einer Schleife über alle Kombinationen
- `grid.best_score_` ist leicht optimistisch (Selection Bias)
- `grid.best_estimator_` ist das fertig trainierte finale Modell -- direkt einsatzbereit
- Der Test-Score auf `X_test` ist die einzig ehrliche Schlussbeurteilung
- Die `__`-Syntax im `param_grid` adressiert Parameter innerhalb einer Pipeline
